# Baseline — Qwen2.5-VL-3B (zero-shot) on ScreenSpot-V2

Smoke test for the shared infrastructure. Run this once before any task notebook.

**Hypothesis to confirm:** unadapted Qwen2.5-VL-3B is around 27% on ScreenSpot-V2 (per the proposal, §3). If this notebook reports something within ±5 pp of that, the eval harness is wired correctly.

In [ ]:
# Colab/Kaggle bootstrap. Skip when running locally with `make sync`.
import os, sys
if 'google.colab' in sys.modules:
    !pip install -q uv
    !git clone --depth 1 https://github.com/AIS5/action-learning-ais5.git /content/ais5 2>/dev/null || true
    %cd /content/ais5
    !uv sync --extra notebook

In [ ]:
from ais5.data import load_benchmark
from ais5.eval import evaluate_model, by_target_size, by_ui_type
from ais5.models import get_model
from ais5.utils import set_global_seed, setup_logging

setup_logging()
set_global_seed(42)

In [ ]:
model = get_model('qwen2.5-vl-3b')
samples = list(load_benchmark('screenspot-v2'))
print(f'{len(samples)} samples loaded')
print(f'first sample: {samples[0]}')

In [ ]:
# Limit to 50 for a quick sanity run; remove `limit=` for the full eval.
run = evaluate_model(model, samples, benchmark='screenspot-v2', limit=50)
print(f'accuracy = {run.accuracy:.4f} (n={len(run.results)})')
print(f'avg latency = {run.avg_latency_ms:.1f} ms' if run.avg_latency_ms else '')

In [ ]:
by_target_size(run.results)

In [ ]:
by_ui_type(run.results)

## Sanity checks

- Did the parser succeed on most samples? Look for `<click>` patterns in `run.results[i].raw_response`.
- Are the wrong-answer points falling near the bbox or far away? Far → prompt issue. Near → resolution issue.
- Is the average latency reasonable (~1-3 s on Colab T4)? Much higher → check `device_map`.